In [1]:
import torch
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm import tqdm
from pathlib import Path
import os

In [2]:
# Make torch deterministic
_ = torch.manual_seed(0)

In [3]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])

# Load the MNIST dataset
mnist_trainset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
# Create a dataloader for the training
train_loader = torch.utils.data.DataLoader(mnist_trainset, batch_size=10, shuffle=True)

# Load the MNIST test set
mnist_testset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
test_loader = torch.utils.data.DataLoader(mnist_testset, batch_size=10, shuffle=True)

# Define the device
device = "cpu"

100%|██████████| 9.91M/9.91M [00:00<00:00, 17.5MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 470kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.42MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 5.10MB/s]


In [28]:
class VerySimpleNet(nn.Module):
    def __init__(self, hidden_size_1=100, hidden_size_2=100):
        super(VerySimpleNet,self).__init__()
        self.conv1 = nn.Conv2d(1,32,3,1,1)
        self.maxpool1 = nn.MaxPool2d(3,1,1)
        self.linear1 = nn.Linear(28*28*32, hidden_size_1)
        self.linear2 = nn.Linear(hidden_size_1, 10)
        self.relu = nn.ReLU()

    def forward(self, img):

        x = img.view(-1, 1,28,28)
        x = self.conv1(x)
        x = self.maxpool1(x)
        x = x.view(x.size(0), -1)
        x = self.relu(self.linear1(x))
        x = self.linear2(x)
        return x

In [29]:
net = VerySimpleNet().to(device)

In [30]:
print(net)

VerySimpleNet(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (maxpool1): MaxPool2d(kernel_size=3, stride=1, padding=1, dilation=1, ceil_mode=False)
  (linear1): Linear(in_features=25088, out_features=100, bias=True)
  (linear2): Linear(in_features=100, out_features=10, bias=True)
  (relu): ReLU()
)


In [31]:
def train(train_loader, net, epochs=5, total_iterations_limit=None):
    cross_el = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(net.parameters(), lr=0.001)

    total_iterations = 0

    for epoch in range(epochs):
        net.train()

        loss_sum = 0
        num_iterations = 0

        data_iterator = tqdm(train_loader, desc=f'Epoch {epoch+1}')
        if total_iterations_limit is not None:
            data_iterator.total = total_iterations_limit
        for data in data_iterator:
            num_iterations += 1
            total_iterations += 1
            x, y = data
            x = x.to(device)
            y = y.to(device)
            optimizer.zero_grad()

            output = net(x)
            loss = cross_el(output, y)
            loss_sum += loss.item()
            avg_loss = loss_sum / num_iterations
            data_iterator.set_postfix(loss=avg_loss)
            loss.backward()
            optimizer.step()

            if total_iterations_limit is not None and total_iterations >= total_iterations_limit:
                return

def print_size_of_model(model):
    torch.save(model.state_dict(), "temp_delme.p")
    print('Size (KB):', os.path.getsize("temp_delme.p")/1e3)
    os.remove('temp_delme.p')

MODEL_FILENAME = 'simplenet_ptq.pt'

if Path(MODEL_FILENAME).exists():
    net.load_state_dict(torch.load(MODEL_FILENAME))
    print('Loaded model from disk')
else:
    train(train_loader, net, epochs=1)
    # Save the model to disk
    torch.save(net.state_dict(), MODEL_FILENAME)

Epoch 1: 100%|██████████| 6000/6000 [06:18<00:00, 15.85it/s, loss=0.168]


In [32]:
def test(model: nn.Module, total_iterations: int = None):
    correct = 0
    total = 0

    iterations = 0

    model.eval()

    with torch.no_grad():
        for data in tqdm(test_loader, desc='Testing'):
            x, y = data
            x = x.to(device)
            y = y.to(device)
            output = model(x)
            for idx, i in enumerate(output):
                if torch.argmax(i) == y[idx]:
                    correct +=1
                total +=1
            iterations += 1
            if total_iterations is not None and iterations >= total_iterations:
                break
    print(f'Accuracy: {round(correct/total, 3)}')

In [34]:
# Print the weights matrix of the model before quantization
print('Weights before quantization')
print(net.linear1.weight)
print(net.linear1.weight.dtype)
print(net.linear1.weight.shape)

Weights before quantization
Parameter containing:
tensor([[ 0.0119,  0.0058,  0.0041,  ...,  0.0442,  0.0136,  0.0167],
        [-0.0026,  0.0068,  0.0044,  ...,  0.0342, -0.0020,  0.0008],
        [ 0.0324,  0.0335,  0.0365,  ...,  0.0658, -0.0329, -0.0292],
        ...,
        [ 0.0428,  0.0448,  0.0460,  ...,  0.0861, -0.0058, -0.0051],
        [-0.0028, -0.0066, -0.0006,  ...,  0.0192,  0.0102,  0.0106],
        [-0.0038, -0.0010, -0.0069,  ...,  0.0040, -0.0005,  0.0025]],
       requires_grad=True)
torch.float32
torch.Size([100, 25088])


In [35]:
print('Size of the model before quantization')
print_size_of_model(net)

Size of the model before quantization
Size (KB): 10043.558


In [36]:
print(f'Accuracy of the model before quantization: ')
test(net)

Accuracy of the model before quantization: 


Testing: 100%|██████████| 1000/1000 [00:15<00:00, 62.86it/s]

Accuracy: 0.974


Before quantization, after training for a epoch, we have an accuracy of 97.4 percent on MNIST dataset

In [55]:
# Quantized model
class QuantizedVerySimpleNet(nn.Module):
    def __init__(self, hidden_size_1=100, hidden_size_2=100):
        super(QuantizedVerySimpleNet,self).__init__()
        self.quant = torch.quantization.QuantStub()
        self.conv1 = nn.Conv2d(1,32,3,1,1)
        self.maxpool1 = nn.MaxPool2d(3,1,1)
        self.linear1 = nn.Linear(28*28*32, hidden_size_1)
        self.linear2 = nn.Linear(hidden_size_1, 10)
        self.relu = nn.ReLU()
        self.dequant = torch.quantization.DeQuantStub()

    def forward(self, img):

        x = img.view(-1, 1,28,28)
        x = self.quant(x)
        x = self.conv1(x)
        x = self.maxpool1(x)
        x = x.reshape(x.size(0), -1)
        x = self.relu(self.linear1(x))
        x = self.linear2(x)
        x = self.dequant(x)
        return x


In [56]:
net_quantized = QuantizedVerySimpleNet().to(device)
# Copy weights from unquantized model
net_quantized.load_state_dict(net.state_dict())
net_quantized.eval()

QuantizedVerySimpleNet(
  (quant): QuantStub()
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (maxpool1): MaxPool2d(kernel_size=3, stride=1, padding=1, dilation=1, ceil_mode=False)
  (linear1): Linear(in_features=25088, out_features=100, bias=True)
  (linear2): Linear(in_features=100, out_features=10, bias=True)
  (relu): ReLU()
  (dequant): DeQuantStub()
)

In [57]:
net_quantized.qconfig = torch.quantization.get_default_qconfig('fbgemm')
net_quantized = torch.ao.quantization.prepare(net_quantized) # insert observers to capture statistics
net_quantized


/usr/local/lib/python3.11/dist-packages/torch/ao/quantization/observer.py:229: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  warnings.warn(


QuantizedVerySimpleNet(
  (quant): QuantStub(
    (activation_post_process): HistogramObserver(min_val=inf, max_val=-inf)
  )
  (conv1): Conv2d(
    1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)
    (activation_post_process): HistogramObserver(min_val=inf, max_val=-inf)
  )
  (maxpool1): MaxPool2d(kernel_size=3, stride=1, padding=1, dilation=1, ceil_mode=False)
  (linear1): Linear(
    in_features=25088, out_features=100, bias=True
    (activation_post_process): HistogramObserver(min_val=inf, max_val=-inf)
  )
  (linear2): Linear(
    in_features=100, out_features=10, bias=True
    (activation_post_process): HistogramObserver(min_val=inf, max_val=-inf)
  )
  (relu): ReLU()
  (dequant): DeQuantStub()
)

We see that currently the min_val max_val are all inf or -inf. the model needs to collect statistics for its Quant Stubs

In [58]:
test(net_quantized)

Testing: 100%|██████████| 1000/1000 [00:19<00:00, 50.40it/s]

Accuracy: 0.974


In [59]:
print(f'Check statistics of the various layers')
net_quantized

Check statistics of the various layers


QuantizedVerySimpleNet(
  (quant): QuantStub(
    (activation_post_process): HistogramObserver(min_val=-0.4242129623889923, max_val=2.821486711502075)
  )
  (conv1): Conv2d(
    1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)
    (activation_post_process): HistogramObserver(min_val=-3.8997178077697754, max_val=2.5301249027252197)
  )
  (maxpool1): MaxPool2d(kernel_size=3, stride=1, padding=1, dilation=1, ceil_mode=False)
  (linear1): Linear(
    in_features=25088, out_features=100, bias=True
    (activation_post_process): HistogramObserver(min_val=-161.56582641601562, max_val=102.75106811523438)
  )
  (linear2): Linear(
    in_features=100, out_features=10, bias=True
    (activation_post_process): HistogramObserver(min_val=-28.442617416381836, max_val=24.500425338745117)
  )
  (relu): ReLU()
  (dequant): DeQuantStub()
)

In [61]:
# using the statistics collected above, convert the model into a quantized one
net_quantized = torch.ao.quantization.convert(net_quantized)

In [62]:
print(f'Check statistics of the various layers')
net_quantized

Check statistics of the various layers


QuantizedVerySimpleNet(
  (quant): Quantize(scale=tensor([0.0255]), zero_point=tensor([17]), dtype=torch.quint8)
  (conv1): QuantizedConv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), scale=0.03938061371445656, zero_point=78, padding=(1, 1))
  (maxpool1): MaxPool2d(kernel_size=3, stride=1, padding=1, dilation=1, ceil_mode=False)
  (linear1): QuantizedLinear(in_features=25088, out_features=100, scale=1.7194581031799316, zero_point=78, qscheme=torch.per_channel_affine)
  (linear2): QuantizedLinear(in_features=100, out_features=10, scale=0.40425413846969604, zero_point=70, qscheme=torch.per_channel_affine)
  (relu): ReLU()
  (dequant): DeQuantize()
)

In [63]:
# Print the weights matrix of the model after quantization
print('Weights after quantization')
print(torch.int_repr(net_quantized.linear1.weight()))

Weights after quantization
tensor([[ 11,   5,   4,  ...,  42,  13,  16],
        [ -2,   6,   4,  ...,  32,  -2,   1],
        [ 25,  26,  28,  ...,  51, -26, -23],
        ...,
        [ 31,  32,  33,  ...,  62,  -4,  -4],
        [ -3,  -8,  -1,  ...,  23,  12,  13],
        [ -6,  -2, -10,  ...,   6,  -1,   4]], dtype=torch.int8)


In [64]:
print('Original weights: ')
print(net.linear1.weight)
print('')
print(f'Dequantized weights: ')
print(torch.dequantize(net_quantized.linear1.weight()))
print('')
print("mean = ", torch.mean((torch.dequantize(net_quantized.linear1.weight()) - net.linear1.weight)**2))

Original weights: 
Parameter containing:
tensor([[ 0.0119,  0.0058,  0.0041,  ...,  0.0442,  0.0136,  0.0167],
        [-0.0026,  0.0068,  0.0044,  ...,  0.0342, -0.0020,  0.0008],
        [ 0.0324,  0.0335,  0.0365,  ...,  0.0658, -0.0329, -0.0292],
        ...,
        [ 0.0428,  0.0448,  0.0460,  ...,  0.0861, -0.0058, -0.0051],
        [-0.0028, -0.0066, -0.0006,  ...,  0.0192,  0.0102,  0.0106],
        [-0.0038, -0.0010, -0.0069,  ...,  0.0040, -0.0005,  0.0025]],
       requires_grad=True)

Dequantized weights: 
tensor([[ 0.0116,  0.0053,  0.0042,  ...,  0.0443,  0.0137,  0.0169],
        [-0.0021,  0.0064,  0.0043,  ...,  0.0343, -0.0021,  0.0011],
        [ 0.0322,  0.0335,  0.0360,  ...,  0.0656, -0.0335, -0.0296],
        ...,
        [ 0.0431,  0.0445,  0.0459,  ...,  0.0863, -0.0056, -0.0056],
        [-0.0025, -0.0066, -0.0008,  ...,  0.0189,  0.0098,  0.0107],
        [-0.0040, -0.0013, -0.0067,  ...,  0.0040, -0.0007,  0.0027]])

mean =  tensor(1.7149e-07, grad_fn=<Mean

In [65]:
print('Size of the model after quantization')

print_size_of_model(net_quantized)

Size of the model after quantization
Size (KB): 2519.33


original size of the model was 10043.558 kb and currently after quantization, it is 2519.33 kb. an improvement by ~3.98X

In [66]:
print('Testing the model after quantization')
test(net_quantized)

Testing the model after quantization


Testing: 100%|██████████| 1000/1000 [00:05<00:00, 193.73it/s]

Accuracy: 0.974


For our model, we do not see a dip in accuracy, since it is a small model and a small dataset. but it is cool that we were able to get the same KPI with reduced memory footprint and faster operations!